# CS4168 Data Mining Project  
## Spotify Tracks Dataset Analysis

This notebook performs:
- Exploratory Data Analysis (EDA)
- Clustering (K-Means & DBSCAN)
- Classification (Predicting Popularity Category)
- Regression (Predicting Popularity Score)

(Needs filling in and more detail)

To Activate .venv on Windows
-Set-ExecutionPolicy -Scope Process -ExecutionPolicy RemoteSigned   
-.venv\Scripts\Activate.ps1

In [ ]:
# imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.impute import SimpleImputer

In [ ]:
# Load de Data

df = pd.read_csv("tracks2026.csv")
df.head()

# 1. Exploratory Data Analysis

In [ ]:
# Basic data inspect

df.info()
df.describe()
df.isnull().sum()

In [ ]:
# Popularity distribution

plt.figure(figsize=(6,4))
sns.histplot(df["popularity"], bins=30)
plt.title("Popularity Distribution")
plt.show()

In [ ]:
# Correlation Heatmap

plt.figure(figsize=(12,8))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

# 2. Clustering Analysis
Goes here

In [ ]:
# Data prep for Clustering

X_cluster = df.drop(columns=["track_genre"])
X_cluster = X_cluster.select_dtypes(include=np.number)

In [ ]:
# Elbow method for K-Means

inertia = []

for k in range(2, 10):
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("kmeans", KMeans(n_clusters=k, random_state=42))
    ])
    
    pipe.fit(X_cluster)
    inertia.append(pipe.named_steps["kmeans"].inertia_)

plt.plot(range(2,10), inertia)
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

In [ ]:
# Fit Final K-Means (Eg - k=3)

kmeans_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(n_clusters=3, random_state=42))
])

kmeans_pipe.fit(X_cluster)

labels = kmeans_pipe.named_steps["kmeans"].labels_

In [ ]:
# Visualise Clusters (w/ PCA)

# Get preprocessing steps from the fitted pipeline
imputer = kmeans_pipe.named_steps["imputer"]
scaler = kmeans_pipe.named_steps["scaler"]

# Apply preprocessing
X_imputed = imputer.transform(X_cluster)
X_scaled = scaler.transform(X_imputed)

# Now PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Plot
plt.figure(figsize=(6,5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels)
plt.title("K-Means Clusters (PCA Projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
# DBScan

dbscan_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("dbscan", DBSCAN(eps=1.5, min_samples=5))
])

dbscan_pipe.fit(X_cluster)

db_labels = dbscan_pipe.named_steps["dbscan"].labels_

print("Unique clusters:", np.unique(db_labels))

# 3. Classification: Predicting Popularity Category

Goes here

In [ ]:
# Create binary popularity variable

median_popularity = df["popularity"].median()

df["popularity_binary"] = (df["popularity"] > median_popularity).astype(int)

print("Median popularity:", median_popularity)
df["popularity_binary"].value_counts()

In [ ]:
# Remove original popularity column

X = df.drop(columns=["popularity", "popularity_binary"])
y = df["popularity_binary"]

# Keep only numeric features
X = X.select_dtypes(include="number")

X.head()

In [ ]:
# Train(Test) Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# Testing 3 differnet classifications

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
logistic_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

rf_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42))
])

gb_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", GradientBoostingClassifier(random_state=42))
])

In [ ]:
# Cross-Validation

from sklearn.model_selection import cross_val_score

models = {
    "Logistic Regression": logistic_pipe,
    "Random Forest": rf_pipe,
    "Gradient Boosting": gb_pipe
}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
    print(name)
    print("Mean CV Accuracy:", scores.mean())
    print()

In [ ]:
# Train best model

best_model = rf_pipe
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

In [ ]:
# Evaluation metrics

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix

import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# Fetuure importance

rf_model = best_model.named_steps["model"]

importances = rf_model.feature_importances_

feature_importance = (
    pd.Series(importances, index=X.columns)
    .sort_values(ascending=False)
    .head(10)
)

feature_importance.plot(kind="barh")
plt.title("Top Features Influencing Popularity Category")
plt.show()

feature_importance

# 4. Classification Discussion

text goes here